# Building Tokenizers Block by Block — Detailed & Explained

This notebook builds **three real tokenizers from scratch** using the 🤗 Tokenizers library:

1. **BERT** — uses the **WordPiece** algorithm
2. **GPT-2** — uses **(byte-level) BPE**
3. **XLNet** — uses **Unigram** (SentencePiece-style)

**The one big idea:** every tokenizer is a **5-stage pipeline**, and you build any tokenizer
just by choosing a different block for each stage:

| Stage | What it does |
|-------|--------------|
| 1. Normalizer | Clean the text (lowercase, strip accents, fix spaces) |
| 2. Pre-tokenizer | Rough first cut into words |
| 3. Model | The subword algorithm (WordPiece / BPE / Unigram) |
| 4. Post-processor | Add special tokens like `[CLS]` / `[SEP]` |
| 5. Decoder | Glue tokens back into readable text |

Every code cell below is explained line by line in the markdown above it.


## 0. Setup — install and import

Run this **once**. `%pip` (not `!pip`) installs into the *same* environment your notebook
kernel uses, which avoids the common "installed but import fails" problem in VS Code.

After it finishes, **restart the kernel once** (restart icon at the top of the notebook),
then run the rest.


In [ ]:
%pip install -U transformers tokenizers datasets

Now the imports. We bring in everything the three builds use:

- `load_dataset` — to download the WikiText-2 training corpus
- the `tokenizers` building blocks — `normalizers`, `pre_tokenizers`, `models`,
  `processors` (post-processors), `decoders`, `trainers`, and the central `Tokenizer` class
- `Regex` — used by the XLNet normalizer
- the `...TokenizerFast` wrapper classes — to plug our finished tokenizer into 🤗 Transformers


In [1]:
from datasets import load_dataset

from tokenizers import (
    decoders,
    models,
    normalizers,
    pre_tokenizers,
    processors,
    trainers,
    Tokenizer,
    Regex,
)

from transformers import (
    PreTrainedTokenizerFast,
    BertTokenizerFast,
    GPT2TokenizerFast,
    XLNetTokenizerFast,
)

print("All imports OK")

All imports OK


## 1. Get a training corpus

A tokenizer learns its vocabulary from example text, called a **corpus**. We use
**WikiText-2** (a small Wikipedia dump) so things run fast.

`get_training_corpus()` is a **generator**: instead of loading all the text into memory at
once, it hands out 1,000 texts at a time, only when asked. We wrap it in a function so we can
call it fresh each time (a bare generator can only be walked through once).


In [5]:
dataset = load_dataset("wikitext", name="wikitext-2-raw-v1", split="train")

def get_training_corpus():
    for i in range(0, len(dataset), 1000):
        yield dataset[i : i + 1000]["text"]

print("Corpus size:", len(dataset), "rows")

HfUriError: Invalid HF URI 'hf://datasets/wikitext@b08601e04326c79dfdd32d625aee71d232d685c3/.huggingface.yaml'. Repository id must be 'namespace/name', got 'wikitext'.

Optional: dump the whole corpus to a plain text file. Some training methods read directly
from a file instead of an iterator. (You can skip this if you only train from the iterator.)


In [ ]:
with open("wikitext-2.txt", "w", encoding="utf-8") as f:
    for i in range(len(dataset)):
        f.write(dataset[i]["text"] + "\n")

print("Wrote wikitext-2.txt")

# Build 1 — BERT tokenizer (WordPiece)

We'll go through all 5 stages in full detail here. The next two builds reuse the same skeleton
and only highlight what changes.


### Stage 0 — Create the Tokenizer with a WordPiece model

`Tokenizer(models.WordPiece(...))` creates an empty tokenizer whose **engine** is WordPiece.

- `unk_token="[UNK]"` tells the model what to output when it meets a character it has never seen.
  (WordPiece needs this; it falls back to `[UNK]` when a word can't be built from the vocabulary.)
- We don't pass a `vocab` because we're going to **train** it ourselves below.


In [ ]:
tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
print(tokenizer)

### Stage 1 — Normalizer (clean the text)

The normalizer tidies raw text before anything else. We build BERT's normalizer **by hand**
by chaining three steps in a `Sequence` (they run in order):

- `NFD()` — a Unicode step that **separates** an accent from its letter (é → e + ´). This must
  come first, otherwise `StripAccents` can't "see" the accent to remove it.
- `Lowercase()` — makes everything lowercase (BERT-uncased ignores capitalization).
- `StripAccents()` — removes the now-separated accent marks.

(There's also a ready-made shortcut: `normalizers.BertNormalizer(lowercase=True)`.)


In [ ]:
tokenizer.normalizer = normalizers.Sequence(
    [normalizers.NFD(), normalizers.Lowercase(), normalizers.StripAccents()]
)

# Test what the normalizer does to a messy string:
print(tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))
# Expected: 'hello how are u?'

### Stage 2 — Pre-tokenizer (rough split into words)

`Whitespace()` splits on **spaces and punctuation**. Each chunk comes back with its
`(start, end)` character offsets — this is where the offset mapping is born.

(Alternatives: `WhitespaceSplit()` splits only on spaces; you can also compose blocks with
`pre_tokenizers.Sequence([...])`.)


In [ ]:
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# See how it cuts a sentence (notice Let's -> Let ' s and the offsets):
print(tokenizer.pre_tokenizer.pre_tokenize_str("Let's test my pre-tokenizer."))

### Stage 3 — Train the WordPiece model

The model is empty; we train it on the corpus with a **matching trainer**.

**Critical detail:** you MUST list your special tokens in the trainer. They don't appear in the
corpus text, so if you don't tell the trainer about them, they won't be added to the vocabulary.

- `vocab_size=25000` — target vocabulary size.
- `special_tokens=[...]` — BERT's five special tokens.
- (Optional: `min_frequency`, or `continuing_subword_prefix` to change the `##` marker.)

This cell does the actual training and may take a minute.


In [ ]:
special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"]
trainer = trainers.WordPieceTrainer(vocab_size=25000, special_tokens=special_tokens)

tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
print("Trained! Vocab size:", tokenizer.get_vocab_size())

Quick test of the raw tokenizer (before adding special tokens). Notice "tokenizer" splits
into `tok` + `##eni` + `##zer` — the `##` marks pieces that continue the previous one.


In [ ]:
encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)
# Expected something like: ['let', "'", 's', 'test', 'this', 'tok', '##eni', '##zer', '.']

### Stage 4 — Post-processor (add `[CLS]` and `[SEP]`)

The raw output has no special tokens yet. A `TemplateProcessing` post-processor inserts them
using a template:

- `$A` = the first sentence, `$B` = the second sentence (for pairs).
- The `:0` / `:1` after each item is its **token-type ID** (which sentence it belongs to — the
  "segment" info BERT uses).
- `single` wraps one sentence as `[CLS] A [SEP]`; `pair` wraps two as `[CLS] A [SEP] B [SEP]`.

We must pass the **IDs** of `[CLS]` and `[SEP]` so the processor can insert the right tokens.


In [ ]:
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")
print("CLS id:", cls_token_id, " SEP id:", sep_token_id)

tokenizer.post_processor = processors.TemplateProcessing(
    single=f"[CLS]:0 $A:0 [SEP]:0",
    pair=f"[CLS]:0 $A:0 [SEP]:0 $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_token_id), ("[SEP]", sep_token_id)],
)

Now `[CLS]` and `[SEP]` appear automatically. On a **pair** of sentences you also get the
`type_ids` (0 for the first sentence, 1 for the second).


In [ ]:
encoding = tokenizer.encode("Let's test this tokenizer.")
print("Single:", encoding.tokens)

encoding = tokenizer.encode("Let's test this tokenizer...", "on a pair of sentences.")
print("Pair tokens:  ", encoding.tokens)
print("Pair type_ids:", encoding.type_ids)

### Stage 5 — Decoder (tokens back to text)

The decoder reverses tokenization. `decoders.WordPiece(prefix="##")` knows to drop the `##`
and rejoin pieces into clean words.


In [ ]:
tokenizer.decoder = decoders.WordPiece(prefix="##")
print(tokenizer.decode(encoding.ids))

### Save, reload, and wrap for 🤗 Transformers

`save()` writes the entire tokenizer (all 5 blocks + vocabulary) to one JSON file.

To use it with the rest of 🤗 Transformers, we **wrap** the raw `Tokenizer` in a
`PreTrainedTokenizerFast`. The wrapper is where we label which token plays which role
(unk / pad / cls / sep / mask) — the raw tokenizer doesn't track those roles by itself.


In [ ]:
tokenizer.save("bert-wordpiece.json")
reloaded = Tokenizer.from_file("bert-wordpiece.json")

wrapped_bert = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
)
# Or, since it matches BERT:  wrapped_bert = BertTokenizerFast(tokenizer_object=tokenizer)

print(wrapped_bert("Let's test this tokenizer.").tokens()
      if hasattr(wrapped_bert("x"), "tokens") else "wrapped OK")

# Build 2 — GPT-2 tokenizer (byte-level BPE)

Same 5-stage skeleton. Here's what changes from BERT:

- **No `unk_token`** — GPT-2 uses **byte-level BPE**: every character is built from the 256
  possible bytes, so an "unknown" character is impossible.
- **No normalizer at all** — GPT-2 skips cleaning entirely.
- **ByteLevel** pre-tokenizer, post-processor, and decoder (spaces shown as `Ġ`).


### Stage 0 + 1 — Create with BPE, skip the normalizer

In [ ]:
tokenizer = Tokenizer(models.BPE())
# No unk_token needed (byte-level BPE), and GPT-2 uses no normalizer, so we set nothing here.

### Stage 2 — ByteLevel pre-tokenizer

`add_prefix_space=False` means: don't automatically add a space at the very start of the text.
Spaces are represented by the `Ġ` symbol (so the original text is perfectly recoverable later).


In [ ]:
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
print(tokenizer.pre_tokenizer.pre_tokenize_str("Let's test pre-tokenization!"))

### Stage 3 — Train with a BPE trainer

GPT-2's only special token is `<|endoftext|>` (it marks where one document ends).


In [ ]:
trainer = trainers.BpeTrainer(vocab_size=25000, special_tokens=["<|endoftext|>"])
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)
# Note the 'Ġ' marking spaces, e.g. 'Ġtest', 'Ġthis'

### Stage 4 + 5 — ByteLevel post-processor and decoder

`trim_offsets=False` keeps the leading space as part of a `Ġ`-token's offset, so slicing the
text for token `Ġtest` returns `' test'` (with the space). The `ByteLevel` decoder turns the
byte representation back into normal text.


In [ ]:
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

# Show that the offset of the 'Ġtest' token (index 4) includes the leading space:
sentence = "Let's test this tokenizer."
encoding = tokenizer.encode(sentence)
start, end = encoding.offsets[4]
print(repr(sentence[start:end]))   # -> ' test'

tokenizer.decoder = decoders.ByteLevel()
print(tokenizer.decode(encoding.ids))  # -> "Let's test this tokenizer." 

### Wrap for 🤗 Transformers

In [ ]:
tokenizer.save("gpt2-bpe.json")

wrapped_gpt2 = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<|endoftext|>",
    eos_token="<|endoftext|>",
)
# Or:  wrapped_gpt2 = GPT2TokenizerFast(tokenizer_object=tokenizer)
print("GPT-2 tokenizer wrapped OK")

# Build 3 — XLNet tokenizer (Unigram / SentencePiece)

Same skeleton again. What's different:

- **SentencePiece-style normalizer** (quote replacements + NFKD + strip accents + collapse spaces).
- **Metaspace** pre-tokenizer (marks spaces with `▁`).
- **Unigram** model — and you MUST pass `unk_token` to its trainer.
- Post-processor puts `<cls>` at the **end** (type id 2), and the tokenizer **pads on the left**.


### Stage 0 + 1 — Create with Unigram, add the SentencePiece normalizer

`Replace("``", '"')` and `Replace("''", '"')` turn typewriter-style double quotes into a normal
`"`. `NFKD` + `StripAccents` remove accents. The `Regex(" {2,}")` rule collapses runs of 2+
spaces into a single space.


In [ ]:
tokenizer = Tokenizer(models.Unigram())

tokenizer.normalizer = normalizers.Sequence(
    [
        normalizers.Replace("``", '"'),
        normalizers.Replace("''", '"'),
        normalizers.NFKD(),
        normalizers.StripAccents(),
        normalizers.Replace(Regex(" {2,}"), " "),
    ]
)

### Stage 2 — Metaspace pre-tokenizer

`Metaspace` is the SentencePiece pre-tokenizer: it keeps spaces and shows them as `▁`,
so the tokenization is reversible.


In [ ]:
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
print(tokenizer.pre_tokenizer.pre_tokenize_str("Let's test the pre-tokenizer!"))

### Stage 3 — Train with a Unigram trainer

**Don't forget `unk_token`** — the Unigram trainer requires it. Optional knobs:
`shrinking_factor` (how much to prune each round, default 0.75) and
`max_piece_length` (longest allowed token, default 16).


In [ ]:
special_tokens = ["<cls>", "<sep>", "<unk>", "<pad>", "<mask>", "<s>", "</s>"]
trainer = trainers.UnigramTrainer(
    vocab_size=25000, special_tokens=special_tokens, unk_token="<unk>"
)
tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)

encoding = tokenizer.encode("Let's test this tokenizer.")
print(encoding.tokens)   # spaces shown as '▁'

### Stage 4 — Post-processor (XLNet's quirk: `<cls>` at the END)

XLNet places `<cls>` at the **end** of the sequence with **type id 2**, and `<sep>` separates
sentences. `$A` / `$B` are the two sentences as before.


In [ ]:
cls_token_id = tokenizer.token_to_id("<cls>")
sep_token_id = tokenizer.token_to_id("<sep>")
print("cls id:", cls_token_id, " sep id:", sep_token_id)

tokenizer.post_processor = processors.TemplateProcessing(
    single="$A:0 <sep>:0 <cls>:2",
    pair="$A:0 <sep>:0 $B:1 <sep>:1 <cls>:2",
    special_tokens=[("<sep>", sep_token_id), ("<cls>", cls_token_id)],
)

encoding = tokenizer.encode("Let's test this tokenizer...", "on a pair of sentences!")
print(encoding.tokens)
print(encoding.type_ids)   # note the trailing 2 for <cls>

### Stage 5 — Metaspace decoder + wrap (with left padding)

The `Metaspace` decoder turns `▁` back into spaces. When wrapping, we must also set
`padding_side="left"` because of XLNet's left-padding behavior.


In [ ]:
tokenizer.decoder = decoders.Metaspace()
print(tokenizer.decode(encoding.ids))

tokenizer.save("xlnet-unigram.json")

wrapped_xlnet = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    cls_token="<cls>",
    sep_token="<sep>",
    mask_token="<mask>",
    padding_side="left",
)
# Or:  wrapped_xlnet = XLNetTokenizerFast(tokenizer_object=tokenizer)
print("XLNet tokenizer wrapped OK")

# Summary — the three builds side by side

| Stage | BERT (WordPiece) | GPT-2 (BPE) | XLNet (Unigram) |
|-------|------------------|-------------|-----------------|
| Normalizer | NFD + Lowercase + StripAccents | **none** | quotes + NFKD + StripAccents + collapse spaces |
| Pre-tokenizer | Whitespace | ByteLevel (`Ġ`) | Metaspace (`▁`) |
| Model | WordPiece (`unk_token`) | BPE (byte-level, no unk) | Unigram (`unk_token`) |
| Trainer | WordPieceTrainer | BpeTrainer | UnigramTrainer |
| Post-processor | `[CLS] A [SEP]` | ByteLevel | `A <sep> <cls>` (cls at end) |
| Decoder | WordPiece(`##`) | ByteLevel | Metaspace |
| Wrap | PreTrainedTokenizerFast / BertTokenizerFast | ... / GPT2TokenizerFast | ... / XLNetTokenizerFast (pad left) |

**The takeaway:** the skeleton is identical every time — five blocks in a row. You build a
different tokenizer simply by dropping a different block into each slot.
